# OpenEar Business Agent (Maguy)

## 0. Setup
Install libraries, load the API key from `.env`, and read the business files.

In [21]:
%pip install -q google-genai python-dotenv pypdf gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pypdf import PdfReader

load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")
assert api_key, "GEMINI_API_KEY not found. Check your .env file."

client = genai.Client(api_key=api_key)
MODEL = "gemini-3.5-flash-lite"

# Load the business knowledge the agent is allowed to use
with open("me/business_summary.txt", encoding="utf-8") as f:
    summary = f.read()

reader = PdfReader("me/about_business.pdf")
profile = "\n".join(page.extract_text() or "" for page in reader.pages)

print(f"Summary: {len(summary)} characters")
print(f"Profile: {len(profile)} characters, {len(reader.pages)} pages")

Summary: 2174 characters
Profile: 4688 characters, 3 pages


In [23]:
# Resilient Gemini call: retry, then fall back to another model on overload (503) or rate limit (429)
import time
from google.genai import errors

FALLBACK_MODELS = [MODEL, "gemini-3.5-flash-lite", "gemini-3.6-flash", "gemini-3.8-flash"]

def generate(contents, config=None, retries=2):
    last_error = None
    for model in dict.fromkeys(FALLBACK_MODELS):  # removes duplicates, keeps order
        for attempt in range(retries):
            try:
                return client.models.generate_content(model=model, contents=contents, config=config)
            except (errors.ServerError, errors.ClientError) as e:
                if getattr(e, "code", None) not in (429, 500, 503):
                    raise  # real errors (bad key, bad request) still surface
                last_error = e
                print(f"[{model} busy ({e.code}), retrying...]")
                time.sleep(2 * (attempt + 1))
    raise last_error

In [5]:
# Quick check that the key and model work (goes through the fallback helper)
response = generate("Reply with exactly: Maguy is online")
print(response.text)

Maguy is online


## 1. Persona
Who the agent is. Shapes tone and style.

In [24]:
# === PERSONA ===
PERSONA = """
You are Maguy, the sales assistant of OpenEar, a Lebanese online store for open-ear sports headphones.
You help athletes find the right headphones for their sport and guide them toward placing an order.
You are concise, friendly and sporty: short answers, no long paragraphs.
You match the customer's tone and language, replying in English, Arabic or Arabizi depending on how they write.
"""

## 2. Instructions
Operational guardrails the agent must follow.

In [25]:
# === INSTRUCTIONS ===
INSTRUCTIONS = """
1. Only use facts from the OpenEar business summary and business profile provided below.
2. Never invent prices, discounts, specs, stock, delivery times or policies.
3. Only answer questions about OpenEar's products, choosing headphones for a sport, orders, delivery, payment,
   warranty, returns and contact details. Politely decline anything else and steer back to OpenEar.
4. If a question is about OpenEar but the answer is not in the files, call record_feedback with the question,
   then tell the customer the team will follow up and share support@openear.store or @openear.lb on Instagram.
5. Before recommending a model, ask which sport the customer trains for if they haven't said.
   Then recommend: Stride for casual running, Pace Pro for serious running, trail and cycling,
   Swim for swimming and triathlon, Ultra for long-distance and ultra events.
6. When a customer wants to buy, order or reserve, or asks about availability, ask for their name and email,
   then call record_customer_interest with a short note of what they want.
7. If a customer showed buying interest but hasn't given an email, ask for it once before the conversation ends.
   Do not insist more than once.
8. Never ask for card numbers or payment details in the chat.
9. Never promise that an order is reserved, confirmed or shipped, and never name a team member who will contact
   the customer. Say the team will email them to confirm the order, delivery and payment.
10. Never assume details the customer didn't give (city, colour, payment method). Ask instead.
"""

## 3. Router
Before Maguy answers, a lightweight Gemini call classifies the message into one of three intents:
- **general**: questions about products, delivery, payment, warranty, returns
- **lead**: wants to buy, order or reserve, asks about availability, or is giving their name/email
- **feedback**: complaints, suggestions, or questions the business files cannot answer

The last two turns of the chat are included, so a reply like "Tatiana, t@mail.com" is recognised as a lead.

In [26]:
INTENTS = ["general", "lead", "feedback"]

ROUTER_PROMPT = """You are a message router for OpenEar, a store selling open-ear sports headphones.
Classify the customer's LATEST message into exactly one intent:
- general: questions about products, specs, prices, delivery, payment, warranty, returns, or choosing a model
- lead: wants to buy, order or reserve, asks about availability, or is giving their name or email
- feedback: a complaint, a suggestion, or a question the store's information cannot answer
  (for example international shipping, discounts, bulk orders, or anything not covered)

Store information:
{knowledge}

Recent conversation:
{recent}

Latest message: {message}

Reply with one word only: general, lead or feedback."""


def route(message, history=None):
    """Classify a message into general / lead / feedback."""
    recent = ""
    for turn in (history or [])[-2:]:
        recent += f"{turn['role']}: {turn['content']}\n"
    prompt = ROUTER_PROMPT.format(knowledge=summary, recent=recent or "(none)", message=message)
    response = generate(prompt, config=types.GenerateContentConfig(temperature=0))
    label = (response.text or "").strip().lower()
    for intent in INTENTS:
        if intent in label:
            return intent
    return "general"  # safe default if the model replies with something unexpected

In [27]:
# Show the router working on sample messages
samples = [
    "What's the battery life of the Pace Pro?",
    "Do you deliver to Tripoli?",
    "I want to order the OpenEar Swim",
    "Tatiana, tatiana@mail.com",
    "Do you ship to Dubai?",
    "Do you have a student discount?",
    "My Stride stopped charging after a week",
]
for msg in samples:
    print(f"{route(msg):9} <- {msg}")

general   <- What's the battery life of the Pace Pro?
general   <- Do you deliver to Tripoli?
lead      <- I want to order the OpenEar Swim
lead      <- Tatiana, tatiana@mail.com
feedback  <- Do you ship to Dubai?
feedback  <- Do you have a student discount?
feedback  <- My Stride stopped charging after a week


## 4. Tools
Maguy has two tools. Each one prints a log line and appends a JSON record to a file in `logs/`, so the owner can review leads and unanswered questions later.

### Tool schema (MCP-style)

| Tool | Description | Parameters | Output |
|---|---|---|---|
| `record_customer_interest` | Saves a lead when a customer wants to buy, order or reserve and has shared their contact details. | `email` (string, required): customer email<br>`name` (string, required): customer name<br>`message` (string, required): what they want, e.g. "Pace Pro, black, marathon training" | `{"status": "recorded", "type": "lead"}` and a line in `logs/leads.jsonl` |
| `record_feedback` | Logs a question Maguy could not answer from the business files, or a complaint or suggestion. | `question` (string, required): the customer's question or feedback, in their words | `{"status": "recorded", "type": "feedback"}` and a line in `logs/feedback.jsonl` |

In [30]:
import json, re
from datetime import datetime

os.makedirs("logs", exist_ok=True)

def _append_log(filename, record):
    record["timestamp"] = datetime.now().isoformat(timespec="seconds")
    with open(os.path.join("logs", filename), "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$")
def record_customer_interest(email, name, message):
    """Log a lead: a customer who wants to buy and shared their contact details."""
    if not EMAIL_RE.match(email.strip()):
        print(f"[LEAD REJECTED] invalid email: {email}")
        return {"status": "error", "reason": "Invalid email address. Ask the customer to re-enter it."}
    print(f"[LEAD] {name} <{email}>: {message}")
    _append_log("leads.jsonl", {"name": name, "email": email, "message": message})
    return {"status": "recorded", "type": "lead"}


def record_feedback(question):
    """Log a question Maguy could not answer, or a complaint or suggestion."""
    print(f"[FEEDBACK] {question}")
    _append_log("feedback.jsonl", {"question": question})
    return {"status": "recorded", "type": "feedback"}


# JSON schemas sent to Gemini so the model knows when and how to call each tool
TOOL_SCHEMAS = [
    {
        "name": "record_customer_interest",
        "description": "Save a lead when a customer wants to buy, order or reserve an OpenEar product "
                       "and has given their name and email.",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "The customer's email address"},
                "name": {"type": "string", "description": "The customer's name"},
                "message": {"type": "string", "description": "What the customer wants, e.g. product, colour, sport"},
            },
            "required": ["email", "name", "message"],
        },
    },
    {
        "name": "record_feedback",
        "description": "Log a customer question that the OpenEar business files cannot answer, "
                       "or a complaint or suggestion, so the team can follow up.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {"type": "string", "description": "The customer's question or feedback, in their words"},
            },
            "required": ["question"],
        },
    },
]

TOOLS = types.Tool(function_declarations=TOOL_SCHEMAS)
TOOL_FUNCTIONS = {
    "record_customer_interest": record_customer_interest,
    "record_feedback": record_feedback,
}
print("Tools ready:", list(TOOL_FUNCTIONS))

Tools ready: ['record_customer_interest', 'record_feedback']


In [31]:
# Call each tool once by hand to check the logging works
print(record_customer_interest("test@example.com", "Test Customer", "Pace Pro, black, marathon training"))
print(record_feedback("Do you ship to Dubai?"))

[LEAD] Test Customer <test@example.com>: Pace Pro, black, marathon training
{'status': 'recorded', 'type': 'lead'}
[FEEDBACK] Do you ship to Dubai?
{'status': 'recorded', 'type': 'feedback'}


## 5. System prompt and chat loop
The system prompt combines the persona, the instructions and the business files.
For every message, the agent routes it, calls Gemini with the history and tool schemas,
runs any tool calls the model makes, sends the results back, and returns the final reply.

In [32]:
SYSTEM_PROMPT = f"""{PERSONA}

## Instructions
{INSTRUCTIONS}

## Business summary
{summary}

## Business profile
{profile}
"""

INTENT_HINTS = {
    "general": "Answer from the business files.",
    "lead": "The customer shows buying interest. If you have their name and email, call record_customer_interest. "
            "Otherwise ask for them. If the tool returns an error, ask the customer to fix that detail.",
    "feedback": "This is feedback or a question the files may not answer. Delivery outside Lebanon, discounts and "
                "bulk orders are NOT covered: always call record_feedback for them, then tell the customer the team "
                "will follow up.",
}


def to_contents(history):
    """Convert [{'role': 'user'|'assistant', 'content': str}] into Gemini contents."""
    return [types.Content(role="model" if t["role"] == "assistant" else "user",
                          parts=[types.Part.from_text(text=t["content"])]) for t in history]


def run_model(contents, config):
    """Call Gemini, run any tool calls, and loop until the model returns text."""
    for _ in range(5):
        response = generate(contents, config=config)
        calls = response.function_calls or []
        if not calls:
            return response.text
        contents.append(response.candidates[0].content)
        results = []
        for call in calls:
            func = TOOL_FUNCTIONS.get(call.name)
            result = func(**dict(call.args)) if func else {"error": f"unknown tool {call.name}"}
            results.append(types.Part.from_function_response(name=call.name, response=result))
        contents.append(types.Content(role="user", parts=results))
    return "Sorry, something went wrong. Please contact support@openear.store."


def chat(message, history, verbose=True):
    """One customer turn: route -> draft (with tools) -> reflect -> rewrite if rejected."""
    intent = route(message, history)
    if verbose:
        print(f"[ROUTER] {intent}")
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT + f"\n## Router intent for this message: {intent}\n{INTENT_HINTS[intent]}",
        tools=[TOOLS],
        temperature=0.4,
    )
    contents = to_contents(history) + [types.Content(role="user", parts=[types.Part.from_text(text=message)])]

    draft = run_model(contents, config)
    verdict = reflect(draft, history, message)
    if verdict.get("approved", True):
        if verbose:
            print("[REFLECT] approved")
        return draft

    if verbose:
        print(f"[REFLECT] rejected: {verdict.get('issues')}")
    contents.append(types.Content(role="model", parts=[types.Part.from_text(text=draft)]))
    contents.append(types.Content(role="user", parts=[types.Part.from_text(
        text=f"[Internal review, not from the customer] Your reply has these problems: {verdict.get('issues')} "
             "Rewrite it to fix them. Do not mention this review.")]))
    rewrite_config = config.model_copy(update={"tools": None})  # no tools on rewrite, so leads aren't logged twice
    return run_model(contents, rewrite_config)

## 6. Reflection (bonus)
Before a reply reaches the customer, a second Gemini call reviews the draft against the business files and the conversation.
If it invents facts, promises a reservation, names a team member, assumes details or accepts a bad email, it is rejected
and Maguy rewrites it once using the reviewer's feedback.

In [33]:
REFLECT_PROMPT = """You review replies written by Maguy, the sales assistant of OpenEar.
Check the DRAFT reply against the business files and the conversation. Reject it if it:
- states a fact that is not in the business files (price, spec, policy, delivery time, discount)
- says an order is reserved, confirmed or shipped, or names a team member who will contact the customer
- assumes something the customer did not say (their city, colour, payment method)
- accepts an email address that is clearly invalid
- talks about topics unrelated to OpenEar

Business files:
{knowledge}

Conversation:
{conversation}

DRAFT reply:
{draft}

Reply in JSON only: {{"approved": true or false, "issues": "short explanation, empty if approved"}}"""


def reflect(draft, history, message):
    """Ask Gemini to review a draft. Returns {'approved': bool, 'issues': str}."""
    conversation = "".join(f"{t['role']}: {t['content']}\n" for t in history[-6:]) + f"user: {message}\n"
    prompt = REFLECT_PROMPT.format(knowledge=summary + "\n\n" + profile, conversation=conversation, draft=draft)
    response = generate(prompt, config=types.GenerateContentConfig(
        temperature=0, response_mime_type="application/json"))
    try:
        return json.loads(response.text)
    except (json.JSONDecodeError, TypeError):
        return {"approved": True, "issues": ""}  # if the reviewer fails, don't block the reply

In [36]:
import gradio as gr

def _text(content):
    """Gradio can send content as a string or as a list of parts; return plain text."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(p.get("text", "") for p in content if isinstance(p, dict))
    return ""

def respond(message, history):
    clean = [{"role": h["role"], "content": _text(h["content"])} for h in history if isinstance(h, dict)]
    return chat(message, clean)

demo = gr.ChatInterface(
    respond,
    title="OpenEar · Chat with Maguy",
    description="Open-ear sports headphones, tested by athletes, delivered across Lebanon.",
    examples=["Which headphones should I get?", "Do you deliver to Tripoli?", "Do you ship to Dubai?"],
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


[ROUTER] general
[REFLECT] approved
[ROUTER] general
[REFLECT] approved
[ROUTER] feedback
[FEEDBACK] Do you ship to dubai
[REFLECT] approved
[ROUTER] lead
[REFLECT] rejected: The draft says 'To reserve your order', which violates the rule against saying an order is reserved, confirmed, or shipped.
[ROUTER] lead
[LEAD REJECTED] invalid email: tatiananohra@
[REFLECT] rejected: The customer provided an invalid/incomplete email address, and the draft accepts/processes it by asking the customer to complete it, which violates the rule against accepting clearly invalid email addresses.
[ROUTER] lead
[LEAD] Tatiana Nohra <tatiananohra5@gmail.com>: Reserve triathlon headphones for Lebanon
[REFLECT] rejected: The draft says the team will confirm the order, which violates the rule against saying an order is reserved, confirmed or shipped, or naming a team member who will contact the customer.
